# IPS — Phase 0 on Kaggle

Attach **IPS** + **EuRoC** datasets, enable **GPU**, then run cells in order.

Full guide: `docs/PHASE0_KAGGLE.md`

In [ ]:
# Step 1 — EDIT these slugs after printing /kaggle/input
import os

for name in sorted(os.listdir("/kaggle/input")):
    print(name, "->", os.listdir(f"/kaggle/input/{name}")[:6])

IPS_SLUG = "ips"  # <-- CHANGE ME
EUROC_SLUG = "euroc"  # <-- CHANGE ME
EUROC_SUBDIR = ""  # e.g. "euroc" if sequences are under euroc/MH_01_easy

IPS_ROOT = f"/kaggle/input/{IPS_SLUG}"
EUROC_ROOT = f"/kaggle/input/{EUROC_SLUG}/{EUROC_SUBDIR}".rstrip("/")
VIFT_ROOT = f"{IPS_ROOT}/third_party/VIFT"
assert os.path.isdir(VIFT_ROOT), VIFT_ROOT
print("OK VIFT:", VIFT_ROOT)

In [ ]:
# Step 2 — Writable copy + EuRoC symlink
import shutil

WORK = "/kaggle/working/IPS"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(IPS_ROOT, WORK)

VIFT = f"{WORK}/third_party/VIFT"
os.makedirs(f"{VIFT}/pretrained_models", exist_ok=True)
euroc_link = f"{VIFT}/data/euroc"
if os.path.lexists(euroc_link):
    os.remove(euroc_link)
os.symlink(EUROC_ROOT, euroc_link)
print("WORK:", WORK)
print("EuRoC:", os.listdir(EUROC_ROOT)[:5])

In [ ]:
# Step 3 — Find VIFT (handles /IPS/IPS/ nested copy)
import os, shutil

def find_vift(base="/kaggle/working/IPS"):
    for root, dirs, files in os.walk(base):
        if "train.py" in files and root.endswith(os.path.join("third_party", "VIFT")):
            return root
    raise FileNotFoundError("VIFT not under " + base)

VIFT = find_vift()
open(os.path.join(VIFT, ".project-root"), "w").close()  # required by rootutils
os.environ["PROJECT_ROOT"] = VIFT
print("VIFT =", VIFT)

# Step 3 — Install deps
import os
os.chdir(VIFT)
!pip install -q -r requirements.txt
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Step 4 — Phase 0 smoke test (dummy_vio)
os.chdir(VIFT)
!python src/train.py experiment=dummy_vio trainer=gpu trainer.max_epochs=2 test=false

In [ ]:
# Step 5 — Phase 1 check: EuRoC GT + one batch
import sys
sys.path.insert(0, "/kaggle/working/IPS/third_party/VIFT")
os.environ["PROJECT_ROOT"] = "/kaggle/working/IPS/third_party/VIFT"

!python /kaggle/working/IPS/scripts/phase1/verify_gt_ate.py \
  --root /kaggle/working/IPS/third_party/VIFT/data/euroc --seq MH_01_easy

from src.data.components.EuRoC_dataset import EuRoC
from src.utils import custom_transform
t = custom_transform.Compose([custom_transform.ToTensor(), custom_transform.Resize((256, 512))])
ds = EuRoC(root="/kaggle/working/IPS/third_party/VIFT/data/euroc", sequences=["MH_01_easy"], transform=t)
(x, y) = ds[0]
print("PASS dataloader", x[0].shape, x[1].shape, y.shape)